In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from PIL import Image

In [2]:
# Setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


In [3]:
from torch.utils.data.sampler import SubsetRandomSampler


# Data augmentation for the training dataset
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229,0.224,0.225])
])

#Loading the dataset
train_folder_path = r'/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train'

train_dataset = ImageFolder(root=train_folder_path, transform=train_transforms)
val_dataset = ImageFolder(root=train_folder_path, transform=val_transforms)


#Making 80/20 split for training and testing
num_train = len(train_dataset)
indices = list(range(num_train))
np.random.shuffle(indices)
split = int(np.floor(0.2*num_train))

train_idx, val_idx = indices[split:], indices[:split]

#Samplers
train_sampler = SubsetRandomSampler(train_idx)
val_sampler = SubsetRandomSampler(val_idx)

#Loaders
train_loader = DataLoader(train_dataset, batch_size=32, sampler=train_sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, sampler=val_sampler, num_workers=2)

print(f"Data Split: {len(train_idx)} Training Images | {len(val_idx)} Validation Images.")

#Imbalanced number of images handler
class_counts = np.bincount(train_dataset.targets)
total_samples = len(train_dataset)
class_weights = total_samples / (len(train_dataset.classes) * class_counts)
weights_tensor = torch.FloatTensor(class_weights).to(device)



Data Split: 10531 Training Images | 2632 Validation Images.


In [4]:
class CNN_V3(nn.Module):
    def __init__(self, num_classes=17):
        super(CNN_V3, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        # ---> THE SECRET WEAPON: Global Average Pooling <---
        # This replaces the massive Flatten and first Linear layer!
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Now the brain only has to connect 512 inputs directly to the 17 classes
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        # Apply GAP instead of Flatten
        x = self.global_pool(x)
        x = x.view(x.size(0), -1) # Flattens the tiny 1x1 map
        
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = CNN_V3(num_classes=17).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# ---> THE SCHEDULER <---
# If Val Loss doesn't improve for 3 epochs, cut the learning rate in half.
# Updated Scheduler without the deprecated 'verbose' argument
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
print("CNN V3 with GAP and LR Scheduler successfully built!")

CNN V3 with GAP and LR Scheduler successfully built!


In [5]:
epochs = 40
best_val_loss = float('inf')
best_epoch = 0

print("Started Training the big boy ツ")

for epoch in range(epochs):
    #TRAINING PHASE
    model.train()
    running_train_loss = 0.0
    for i, (images, labels) in  enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()       #Clear old math
        outputs = model(images)     #Guess the styles
        loss = criterion(outputs, labels) #Check how wrong the model was
        loss.backward()             #Backtrack to error source and see how to fix it
        optimizer.step()            #Update the brain

        running_train_loss += loss.item()
    

        #Printing a little update every 100 batch to check that my laptop wasn't fried :>
        if (i+1)%100 == 0:
            print(f"  Batch {i+1}/{len(train_loader)}")
    avg_train_loss = running_train_loss/len(train_loader)

    # VALIDATION PHASE
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            #Calculating the accuracy percentage

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    # Grab the current Learning Rate from the optimizer's engine room
    current_lr = optimizer.param_groups[0]['lr']
    
    # Print the summary, now including the exact LR being used!
    print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")

    #CHECKPOINT
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_interior_design_model.pth')
        best_epoch = epoch + 1
        print(f"  --> New Best Model! (Peaked at Epoch {best_epoch}) Saved to disk.")
    scheduler.step(avg_val_loss)

print("-" * 50)
print(f"Training finished! The ultimate best version of your model was saved from Epoch {best_epoch}.")
print("MODEL IS NOW TRAINED HOORAY")

Started Training the big boy ツ
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [1/40] | LR: 0.001000 | Train Loss: 2.7632 | Val Loss: 2.6510 | Val Accuracy: 14.29%
  --> New Best Model! (Peaked at Epoch 1) Saved to disk.
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [2/40] | LR: 0.001000 | Train Loss: 2.6641 | Val Loss: 2.6158 | Val Accuracy: 16.11%
  --> New Best Model! (Peaked at Epoch 2) Saved to disk.
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [3/40] | LR: 0.001000 | Train Loss: 2.6067 | Val Loss: 2.5301 | Val Accuracy: 19.00%
  --> New Best Model! (Peaked at Epoch 3) Saved to disk.
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [4/40] | LR: 0.001000 | Train Loss: 2.5629 | Val Loss: 2.5862 | Val Accuracy: 18.20%
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [5/40] | LR: 0.001000 | Train Loss: 2.4965 | Val Loss: 2.6042 | Val Accuracy: 19.76%
  Batch 100/330
  Batch 200/330
  Batch 300/330
Epoch [6/40] | LR: 0.001000 | Train Loss: 2.4214 | Val 

In [6]:
class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.folder_path, img_name)
        
        try:
            # Try to open the image normally
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # IF IT FAILS: Print a warning and create a fake black image to prevent a crash
            print(f"  [!] Warning: {img_name} is corrupted. Using a blank placeholder.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_name
    
# Setting up the testing data
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_folder_path = r'/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test'

test_dataset = TestDataset(folder_path=test_folder_path, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

model.eval() #IMPORTANT: Turns off the Dropout constraint for maximum testing power
predictions = []

print(f"Starting inference on {len(test_dataset)} test images...")

with torch.no_grad():
    for images, img_names in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for img_name, pred in zip(img_names, predicted):
            predictions.append({'ImageName': img_name, 'ClassLabel':pred.item()})
        if (i+1) % 25 == 0:
            print(f"  Processed batch {i+1}/{len(test_loader)}...")

# 4. Generate the CSV file
submission_df = pd.DataFrame(predictions)

# ---> THE SORTING UPGRADE <---
# 1. Extract the number from the string using a Regular Expression and convert to integer
submission_df['sort_number'] = submission_df['ImageName'].str.extract(r'(\d+)').astype(int)

# 2. Sort the DataFrame using our new integer column
submission_df = submission_df.sort_values(by='sort_number')

# 3. Drop the temporary sorting column so it doesn't end up in the final CSV
submission_df = submission_df.drop(columns=['sort_number'])

# Save to disk
submission_df.to_csv('Eyad_model_submission_V6.csv', index=False)
print("SUCCESS! Output saved to 'my_final_submission_sorted.csv' in perfect numerical order.")

Starting inference on 5482 test images...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [!] Warning: testimage_4180.jpg is corrupted. Using a blank placeholder.
  [!] Warning: testimage_1881.jpg is corrupted. Using a blank placeholder.
  [!] Warning: testimage_3427.jpg is corrupted. Using a blank placeholder.
  [!] Warning: testimage_3341.jpg is corrupted. Using a blank placeholder.
SUCCESS! Output saved to 'my_final_submission_sorted.csv' in perfect numerical order.
